# Reverse Logistics — Component Inspection Agent (Embedded Gradio App)

This notebook implements the **Cleaning & Inspection** step of a remanufacturing reverse-logistics
flow: components are photographed, a trained image classifier flags each one **Defective** or
**Good Area**, a deterministic decision rule converts that into Pass/Reject, and a multi-agent
chat layer explains results, reports batch QC stats, and gives repair guidance \u2014 grounded in the
real classification output, not invented.

**Dataset:** [Real-life Industrial Dataset of Casting Product](https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product)
(submersible pump impeller castings, labeled `def_front` / `ok_front`). We train a small image
classifier on it directly, instead of asking a vision LLM to guess defects zero-shot.

**Approach:**
1. Download the Kaggle dataset and train a binary image classifier (transfer learning on MobileNetV2)
   to distinguish defective vs. good castings.
2. **Define the label mapping after training** \u2014 Keras assigns label indices from the dataset's
   own folder names (`def_front`, `ok_front`); only once training is done do we map those to the
   app's domain labels (`Defective`, `Good Area`) and decide what each means for Pass/Reject.
3. Save the trained model + label mapping to disk, then build an embedded multi-agent Gradio app
   around it: upload images \u2192 classifier predicts \u2192 rule-based decision \u2192 chat agents (powered
   by Gemini's free tier) explain results, report stats, and suggest repairs using RAG-grounded
   QC guidance.

**Important limitation:** unlike a zero-shot vision LLM, this trained classifier only generalizes to
images that resemble its training distribution (similar casting type, lighting, framing). It will not
reliably classify arbitrary photos of unrelated objects \u2014 retrain on a matching dataset for other
component types.

Fully self-contained: no `git clone`, no dependency on a GitHub repo being reachable. Every module's
source code is embedded directly below and written to disk when you run the cells.


In [ ]:
# 1. Create a working directory (everything below uses paths relative to this)
import os
os.makedirs("/content/reverse_logistics_agent", exist_ok=True)
os.makedirs("/content/reverse_logistics_agent/agents", exist_ok=True)
os.makedirs("/content/reverse_logistics_agent/data", exist_ok=True)
os.makedirs("/content/reverse_logistics_agent/data/uploads", exist_ok=True)
os.makedirs("/content/reverse_logistics_agent/data/models", exist_ok=True)
%cd /content/reverse_logistics_agent


## Part 1 — Download the dataset and train a defect classifier

GPU is recommended for this part (Runtime \u2192 Change runtime type \u2192 GPU) but not required \u2014
the backbone is frozen, so only a small head is actually trained.


In [ ]:
# 2. Download the casting-defect dataset from Kaggle
!pip install -q kagglehub
import kagglehub

dataset_path = kagglehub.dataset_download("ravirajsinh45/real-life-industrial-dataset-of-casting-product")
print("Path to dataset files:", dataset_path)


In [ ]:
# 3. Locate the train/ and test/ directories regardless of the exact nesting kagglehub gives us
import glob
import os

def find_dir(root, name):
    matches = [m for m in glob.glob(os.path.join(root, "**", name), recursive=True) if os.path.isdir(m)]
    if not matches:
        raise FileNotFoundError(f"Could not find a '{name}' directory under {root}")
    return sorted(matches, key=len)[0]

train_dir = find_dir(dataset_path, "train")
test_dir = find_dir(dataset_path, "test")
print("train_dir:", train_dir)
print("test_dir:", test_dir)

for split_dir in (train_dir, test_dir):
    for sub in sorted(os.listdir(split_dir)):
        sub_path = os.path.join(split_dir, sub)
        if os.path.isdir(sub_path):
            n_images = len(os.listdir(sub_path))
            print(f"  {split_dir} / {sub}: {n_images} images")


In [ ]:
# 4. Peek at a few example images from each class
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for row, sub in enumerate(sorted(os.listdir(train_dir))):
    sub_path = os.path.join(train_dir, sub)
    if not os.path.isdir(sub_path):
        continue
    sample_files = sorted(os.listdir(sub_path))[:4]
    for col, fname in enumerate(sample_files):
        img = Image.open(os.path.join(sub_path, fname))
        axes[row, col].imshow(img)
        axes[row, col].set_title(sub, fontsize=10)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# 5. Build train/validation/test datasets
import tensorflow as tf

IMG_SIZE = (160, 160)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
print("Class names (alphabetical order = label indices 0, 1):", class_names)


In [ ]:
# 6. Build the model: MobileNetV2 backbone (frozen) + a small trainable head
# Use layers.Rescaling instead of Lambda(preprocess_input) — Rescaling is natively
# serializable in Keras 3 and avoids Lambda deserialization errors on model reload.
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = layers.Rescaling(scale=1./127.5, offset=-1)(inputs)  # [0,255] -> [-1,1], same as MobileNetV2 preprocess_input
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


In [ ]:
# 7. Train (only the head trains; the MobileNetV2 backbone stays frozen)
EPOCHS = 5
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)


In [ ]:
# 8. Plot training curves
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss")
plt.legend()
plt.show()


In [ ]:
# 9. Evaluate on the held-out test split
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")


## Part 2 — Define the label mapping (after training, not before)

`image_dataset_from_directory` assigned label indices purely from the dataset's own folder names
in alphabetical order (`def_front` \u2192 0, `ok_front` \u2192 1). Only now, after training, do we decide
what those raw folder names mean for the app: the domain-friendly category name, and the Pass/Reject
decision that follows from it.


In [ ]:
# 10. Map the dataset's raw class names to the app's category names and decision rule
print("Keras class names (alphabetical order = label indices 0, 1):", class_names)

LABEL_MAP = {
    "def_front": "Defective",
    "ok_front": "Good Area",
}

DECISION_RULE = {
    "Defective": {"decision": "TIDAK LULUS", "action": "Reject / Repair"},
    "Good Area": {"decision": "LULUS", "action": "Lanjut Assembly"},
}

print("Label mapping:", LABEL_MAP)
print("Decision rule:", DECISION_RULE)


In [ ]:
# 11. Persist the trained model and the label mapping so the agent app below can load them
import json
import os

os.makedirs("data/models", exist_ok=True)
model.save("data/models/casting_defect_classifier.keras")

with open("data/models/label_map.json", "w") as f:
    json.dump({
        "class_names": class_names,
        "label_map": LABEL_MAP,
        "decision_rule": DECISION_RULE,
        "image_size": list(IMG_SIZE),
    }, f, indent=2)

print("\u2713 Saved trained model and label mapping to data/models/")


## Part 3 — Embedded Gradio App

The rest of this notebook is fully self-contained: no `git clone`, no dependency on a GitHub repo
being reachable. Every module's source code is embedded directly below and written to disk when
you run the cells. Detection now comes from the classifier trained above; Gemini's free tier is
only used for the text-reasoning agents (explaining results, summarizing batches, suggesting
repairs) and for RAG embeddings \u2014 not for per-image inference, so there's no per-image API cost
or rate limit to worry about.


In [ ]:
%%writefile requirements.txt
# Core LangChain ecosystem
langchain>=0.3.0,<0.4.0
langchain-core>=0.3.0,<0.4.0
langchain-community>=0.3.0,<0.4.0
langchain-openai>=0.2.0,<0.3.0
langgraph>=0.2.0,<0.3.0
langsmith>=0.1.100,<0.2.0

# Gemini (via OpenAI-compatible endpoint, free tier; used for chat agents + RAG embeddings only)
openai>=1.45.0,<2.0.0

# Vector store
faiss-cpu>=1.8.0,<2.0.0

# UI
gradio

# Utilities
python-dotenv>=1.0.0
pydantic>=2.9.0,<3.0.0
pydantic-settings>=2.5.0
tiktoken>=0.8.0
pandas>=2.0.0
numpy>=1.26.0,<2.0.0
Pillow>=10.0.0
SQLAlchemy>=2.0.0
grandalf

# Note: tensorflow is not pinned here \u2014 Colab ships with it pre-installed.
# If running outside Colab, `pip install tensorflow` before using vision_tools.py.


In [ ]:
# 12. Install the agent-app dependencies
# pydantic<2.11 is required: newer pydantic breaks gradio's API schema introspection
!pip install -q -r requirements.txt
!pip install -q "pydantic<2.11"


## Get a free Gemini API key

1. Go to https://aistudio.google.com/apikey
2. Click "Create API key" and copy it.
3. Run the next cell **on its own** (not via "Run all") and wait for the input box to appear at the top before pasting. The cell after it does a live test call so you'll know right away if the key works.


In [ ]:
# 13. Configure environment variables
import getpass

while True:
    google_api_key = getpass.getpass("Enter your Google AI Studio API key: ").strip()
    if not google_api_key:
        print("\u274c Empty input \u2014 the key box may not have been ready. Try again.")
        continue
    break

env_content = f"""GOOGLE_API_KEY={google_api_key}
LANGSMITH_API_KEY=your_key_here
LANGCHAIN_PROJECT=reverse-logistics-inspection
LANGCHAIN_TRACING_V2=false
LANGSMITH_TRACING=False
LANGSMITH_ENDPOINT=https://api.smith.langchain.com/
LANGSMITH_PROJECT=reverse-logistics-inspection
"""

with open(".env", "w") as f:
    f.write(env_content)

print(f"\u2713 .env saved (key length: {len(google_api_key)}). Run the next cell to verify it actually works.")


In [ ]:
# 13b. Sanity-check the key actually works before initializing anything
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
try:
    _resp = _client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": "Say OK"}],
    )
    print("\u2713 Key works:", _resp.choices[0].message.content)
except Exception as e:
    print("\u274c Key test failed:", e)
    print("Re-run the previous cell and paste the key again.")


## Write the application source files

Each cell below writes one module of the reverse logistics inspection agent application.


In [ ]:
%%writefile config.py
"""Configuration module for environment variables and settings."""
import os
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables
load_dotenv()

# Gemini Configuration (used via Google's OpenAI-compatible endpoint, free tier).
# Only used by the text-reasoning agents and RAG embeddings below \u2014 image
# classification uses the model trained earlier in this notebook, not Gemini.
OPENAI_API_KEY = os.getenv("GOOGLE_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# LangSmith Configuration
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "reverse-logistics-inspection")
LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"

# Database Configuration
DB_PATH = Path("data/reverse_logistics.db")
DB_PATH.parent.mkdir(exist_ok=True)

# Upload directory for images coming in through the Gradio UI
UPLOAD_DIR = Path("data/uploads")
UPLOAD_DIR.mkdir(exist_ok=True)

# Trained classifier + label mapping, saved by the training cells earlier in this notebook
MODEL_PATH = Path("data/models/casting_defect_classifier.keras")
LABEL_MAP_PATH = Path("data/models/label_map.json")

# RAG Configuration
RAG_DOCUMENTS_PATH = Path("data/qc_docs")
RAG_DOCUMENTS_PATH.mkdir(exist_ok=True)
EMBEDDING_MODEL = "gemini-embedding-001"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K_RESULTS = 3

# Model Configuration for the text agents (chat, routing, RAG) \u2014 unrelated to
# the image classifier above.
LLM_MODEL = "gemini-2.5-flash"
TEMPERATURE = 0.3

if __name__ == "__main__":
    # Test configuration
    print("\u2713 Configuration loaded successfully")
    print(f"  LangSmith Project: {LANGSMITH_PROJECT}")
    print(f"  Database Path: {DB_PATH}")
    print(f"  Model Path: {MODEL_PATH}")
    print(f"  LLM Model: {LLM_MODEL}")


In [ ]:
%%writefile db.py
"""Database module for persistent storage with multi-user support.

Tracks two things: chat conversations (for the agent UI) and per-image
inspection records (for QC traceability across upload batches).
"""
import sqlite3
import json
from datetime import datetime
from contextlib import contextmanager
from typing import List, Dict, Optional, Any
import threading
from config import DB_PATH

# Thread-local storage for database connections
_thread_local = threading.local()

class DatabaseManager:
    """Thread-safe database manager for SQLite operations."""

    def __init__(self, db_path: str = DB_PATH):
        self.db_path = str(db_path)
        self._init_db()

    def _get_connection(self):
        """Get thread-local database connection."""
        if not hasattr(_thread_local, "connection"):
            _thread_local.connection = sqlite3.connect(
                self.db_path,
                timeout=30,  # Wait up to 30s for lock
                check_same_thread=False  # We're managing threads manually
            )
            _thread_local.connection.row_factory = sqlite3.Row
        return _thread_local.connection

    @contextmanager
    def get_cursor(self):
        """Context manager for database cursors with automatic commit/rollback."""
        conn = self._get_connection()
        cursor = conn.cursor()
        try:
            yield cursor
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally:
            cursor.close()

    def _init_db(self):
        """Initialize database tables if they don't exist."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS conversations (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    session_id TEXT NOT NULL,
                    user_query TEXT NOT NULL,
                    assistant_response TEXT NOT NULL,
                    agent_used TEXT,
                    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
                    metadata TEXT
                )
            """)
            cursor.execute("""
                CREATE INDEX IF NOT EXISTS idx_session_timestamp
                ON conversations(session_id, timestamp)
            """)
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS inspections (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    batch_id TEXT NOT NULL,
                    filename TEXT NOT NULL,
                    component_type TEXT,
                    categories_found TEXT,
                    decision TEXT,
                    action TEXT,
                    driving_category TEXT,
                    detections TEXT,
                    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
                )
            """)
            cursor.execute("""
                CREATE INDEX IF NOT EXISTS idx_batch
                ON inspections(batch_id, timestamp)
            """)

    def save_conversation(self, session_id: str, user_query: str,
                         assistant_response: str, agent_used: str = None,
                         metadata: Dict = None):
        """Save a conversation turn to database."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                INSERT INTO conversations
                (session_id, user_query, assistant_response, agent_used, metadata)
                VALUES (?, ?, ?, ?, ?)
            """, (
                session_id,
                user_query,
                assistant_response,
                agent_used,
                json.dumps(metadata) if metadata else None
            ))

    def load_session_history(self, session_id: str, limit: int = 50) -> List[Dict]:
        """Load conversation history for a session."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                SELECT user_query, assistant_response, agent_used, timestamp, metadata
                FROM conversations
                WHERE session_id = ?
                ORDER BY timestamp DESC
                LIMIT ?
            """, (session_id, limit))

            rows = cursor.fetchall()
            return [
                {
                    "user_query": row["user_query"],
                    "assistant_response": row["assistant_response"],
                    "agent_used": row["agent_used"],
                    "timestamp": row["timestamp"],
                    "metadata": json.loads(row["metadata"]) if row["metadata"] else {}
                }
                for row in rows
            ]

    def get_all_sessions(self) -> List[str]:
        """Get all unique session IDs."""
        with self.get_cursor() as cursor:
            cursor.execute("SELECT DISTINCT session_id FROM conversations")
            return [row["session_id"] for row in cursor.fetchall()]

    def delete_session(self, session_id: str):
        """Delete a session and all its conversations."""
        with self.get_cursor() as cursor:
            cursor.execute("DELETE FROM conversations WHERE session_id = ?", (session_id,))

    def save_inspection(self, batch_id: str, filename: str, component_type: str,
                        categories_found: List[str], decision: str, action: str,
                        driving_category: str, detections: List[Dict[str, Any]]):
        """Save a single image's inspection result to the database."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                INSERT INTO inspections
                (batch_id, filename, component_type, categories_found, decision, action, driving_category, detections)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                batch_id, filename, component_type,
                json.dumps(categories_found), decision, action, driving_category,
                json.dumps(detections)
            ))

    def get_batch_inspections(self, batch_id: str) -> List[Dict[str, Any]]:
        """Fetch all inspection records belonging to a batch, oldest first."""
        with self.get_cursor() as cursor:
            cursor.execute("""
                SELECT filename, component_type, categories_found, decision, action, driving_category, detections, timestamp
                FROM inspections
                WHERE batch_id = ?
                ORDER BY timestamp ASC
            """, (batch_id,))
            rows = cursor.fetchall()
            return [
                {
                    "filename": row["filename"],
                    "component_type": row["component_type"],
                    "categories_found": json.loads(row["categories_found"]) if row["categories_found"] else [],
                    "decision": row["decision"],
                    "action": row["action"],
                    "driving_category": row["driving_category"],
                    "detections": json.loads(row["detections"]) if row["detections"] else [],
                    "timestamp": row["timestamp"]
                }
                for row in rows
            ]

    def get_latest_batch_id(self) -> Optional[str]:
        """Return the most recently inspected batch id, if any."""
        with self.get_cursor() as cursor:
            cursor.execute("SELECT batch_id FROM inspections ORDER BY timestamp DESC LIMIT 1")
            row = cursor.fetchone()
            return row["batch_id"] if row else None


# Global database instance
db_manager = DatabaseManager()

if __name__ == "__main__":
    # Test database functionality
    print("Testing Database Module...")

    test_batch = "test_batch_123"
    db_manager.save_inspection(
        batch_id=test_batch,
        filename="casting_01.jpeg",
        component_type="submersible pump impeller casting",
        categories_found=["Defective"],
        decision="TIDAK LULUS",
        action="Reject / Repair",
        driving_category="Defective",
        detections=[{"category": "Defective", "confidence": 0.92, "description": "CNN classifier predicted def_front"}]
    )

    records = db_manager.get_batch_inspections(test_batch)
    print(f"\u2713 Saved and loaded {len(records)} inspection records")
    print(f"\u2713 Latest batch id: {db_manager.get_latest_batch_id()}")


In [ ]:
%%writefile vision_tools.py
"""Defect classification tool: loads the CNN trained earlier in this notebook
(data/models/casting_defect_classifier.keras) and runs inference on uploaded images,
then applies a deterministic, rule-based pass/reject decision using the label
mapping that was defined right after training (data/models/label_map.json).
"""
import json
import uuid
from pathlib import Path
from typing import Dict, Any, List, Optional

import numpy as np
import tensorflow as tf

from config import MODEL_PATH, LABEL_MAP_PATH
from db import db_manager

if not MODEL_PATH.exists() or not LABEL_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Trained model ({MODEL_PATH}) or label map ({LABEL_MAP_PATH}) not found. "
        "Run the training cells earlier in this notebook before using this module."
    )

# Model uses a Rescaling layer internally — no custom_objects needed, no manual preprocessing.
_model = tf.keras.models.load_model(MODEL_PATH)
with open(LABEL_MAP_PATH) as f:
    _meta = json.load(f)

CLASS_NAMES = _meta["class_names"]      # e.g. ["def_front", "ok_front"], alphabetical = label order
LABEL_MAP = _meta["label_map"]          # e.g. {"def_front": "Defective", "ok_front": "Good Area"}
DECISION_RULE = _meta["decision_rule"]  # e.g. {"Defective": {...}, "Good Area": {...}}
IMG_SIZE = tuple(_meta["image_size"])   # e.g. (160, 160)

def _load_and_preprocess(image_path: str) -> np.ndarray:
    # Feed raw [0-255] pixels — the model's Rescaling layer handles normalization internally.
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    return np.expand_dims(arr, axis=0)

def classify_component_image(image_path: str) -> Dict[str, Any]:
    """Run the trained CNN on a single image and return its predicted category."""
    batch = _load_and_preprocess(image_path)
    prob_class1 = float(_model.predict(batch, verbose=0)[0][0])  # P(CLASS_NAMES[1])

    if prob_class1 >= 0.5:
        raw_label = CLASS_NAMES[1]
        confidence = prob_class1
    else:
        raw_label = CLASS_NAMES[0]
        confidence = 1.0 - prob_class1

    category = LABEL_MAP.get(raw_label, raw_label)
    return {
        "component_type": "submersible pump impeller casting",
        "detections": [{
            "category": category,
            "confidence": round(confidence, 4),
            "description": f"Trained CNN classifier predicted '{raw_label}' -> '{category}' (p={confidence:.2f})"
        }]
    }

def apply_decision_rule(detections: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Deterministic pass/reject decision."""
    categories_found = sorted({d.get("category", "Good Area") for d in detections}) or ["Good Area"]

    driving_category = categories_found[0]
    for c in categories_found:
        if DECISION_RULE.get(c, {}).get("decision") == "TIDAK LULUS":
            driving_category = c
            break

    rule = DECISION_RULE.get(driving_category, {"decision": "LULUS", "action": "Lanjut Assembly"})
    return {
        "decision": rule["decision"],
        "action": rule["action"],
        "driving_category": driving_category,
        "categories_found": categories_found
    }

def classify_batch(image_paths: List[str], batch_id: Optional[str] = None) -> Dict[str, Any]:
    """Classify a batch of uploaded images and persist each result to the DB."""
    batch_id = batch_id or str(uuid.uuid4())
    records = []

    for image_path in image_paths:
        result = classify_component_image(image_path)
        rule = apply_decision_rule(result["detections"])
        record = {
            "batch_id": batch_id,
            "filename": Path(image_path).name,
            "component_type": result.get("component_type", "unknown"),
            "detections": result["detections"],
            **rule
        }
        db_manager.save_inspection(
            batch_id=batch_id,
            filename=record["filename"],
            component_type=record["component_type"],
            categories_found=record["categories_found"],
            decision=record["decision"],
            action=record["action"],
            driving_category=record["driving_category"],
            detections=record["detections"]
        )
        records.append(record)

    return {"batch_id": batch_id, "records": records}

def get_batch_inspections(batch_id: Optional[str] = None) -> List[Dict[str, Any]]:
    """Fetch all inspection records for a batch (defaults to the most recent batch)."""
    batch_id = batch_id or db_manager.get_latest_batch_id()
    if not batch_id:
        return []
    return db_manager.get_batch_inspections(batch_id)

def get_latest_batch_id() -> Optional[str]:
    """Return the most recently inspected batch id, if any."""
    return db_manager.get_latest_batch_id()

def get_batch_summary(batch_id: Optional[str] = None) -> Dict[str, Any]:
    """Aggregate pass/reject stats and category frequency for a batch."""
    records = get_batch_inspections(batch_id)
    if not records:
        return {"batch_id": batch_id, "n_items": 0}

    n_pass = sum(1 for r in records if r["decision"] == "LULUS")
    n_reject = len(records) - n_pass
    category_counts: Dict[str, int] = {}
    for r in records:
        for c in r["categories_found"]:
            category_counts[c] = category_counts.get(c, 0) + 1

    return {
        "batch_id": batch_id or get_latest_batch_id(),
        "n_items": len(records),
        "n_pass": n_pass,
        "n_reject": n_reject,
        "category_counts": category_counts
    }

if __name__ == "__main__":
    print("Testing decision rule (no model inference needed)...")
    print(apply_decision_rule([{"category": "Defective", "confidence": 0.9}]))
    print(apply_decision_rule([{"category": "Good Area", "confidence": 0.95}]))
    print(f"\nLoaded classes: {CLASS_NAMES}, label map: {LABEL_MAP}")


In [ ]:
%%writefile rag.py
"""RAG module for document retrieval and context injection."""
import os
from typing import List
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langsmith import traceable
from config import (
    OPENAI_API_KEY,
    GEMINI_BASE_URL,
    EMBEDDING_MODEL,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    TOP_K_RESULTS,
)

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=OPENAI_API_KEY,
    base_url=GEMINI_BASE_URL,
    check_embedding_ctx_length=False,
)

class RAGSystem:
    """Lightweight RAG system for reverse logistics / casting QC documents."""

    def __init__(self, persist_directory: str = "data/faiss_index"):
        self.persist_directory = persist_directory
        self.vector_store = None
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
        )
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        """Initialize FAISS vector store from existing index or create new."""
        if os.path.exists(self.persist_directory):
            try:
                self.vector_store = FAISS.load_local(
                    self.persist_directory,
                    embeddings,
                    allow_dangerous_deserialization=True
                )
                print(f"\u2713 Loaded existing FAISS index from {self.persist_directory}")
            except Exception as e:
                print(f"! Could not load existing index: {e}")
                self.vector_store = None

        if self.vector_store is None:
            self._create_sample_documents()

    def _create_sample_documents(self):
        """Create the casting QC knowledge base."""
        documents = [
            Document(
                page_content="""Casting Inspection Process Overview:
                - The flow is Disassembly/Casting -> Cleaning & Inspection -> Salvaging & Machining -> Assembly -> Testing.
                - The trained classifier runs at the Cleaning & Inspection stage and outputs one of two categories per image: Defective or Good Area.
                - Only components classified Good Area continue to Salvaging & Machining; Defective components are pulled for repair, secondary inspection, or scrap.
                - The classifier is binary by design \u2014 it does not by itself say which casting defect subtype is present.""",
                metadata={"category": "process", "source": "process_overview"}
            ),
            Document(
                page_content="""Pass/Reject Decision Rule:
                - Defective -> TIDAK LULUS (Reject / Repair).
                - Good Area -> LULUS (Pass), routed to Lanjut Assembly.
                - This mapping was defined after training, from the dataset's own raw folder labels (def_front -> Defective, ok_front -> Good Area); it is enforced by code, not by the model or an LLM, so it stays auditable.""",
                metadata={"category": "decision_rule", "source": "qc_rules"}
            ),
            Document(
                page_content="""Common Casting Defect Subtypes to Check When Flagged Defective:
                - Blow holes / porosity: trapped gas pockets formed during pouring, often visible as small round voids.
                - Shrinkage defects: irregular cavities from uneven cooling, typically in thicker sections.
                - Sand inclusion: foreign sand particles embedded in the casting surface.
                - Cold shut / misrun: incomplete fill where two streams of metal failed to fuse, leaving a visible seam or gap.
                - Surface cracks: thin fracture lines, often near stress-concentration points.
                - The classifier only says "Defective", not which of these applies \u2014 a human inspector or a secondary zero-shot vision check should identify the subtype before deciding repair vs. scrap.""",
                metadata={"category": "repair_guidance", "source": "defect_subtypes"}
            ),
            Document(
                page_content="""Traceability & QC Reporting:
                - Every inspected image is tied to a batch id and filename in the inspections table so results can be audited later.
                - A batch QC report should include: total items inspected, pass count, reject count, and a breakdown of how often each category appeared.
                - Use traceability records to spot a casting run or supplier batch with an unusually high reject rate, not just to log individual outcomes.""",
                metadata={"category": "traceability", "source": "qc_reporting"}
            ),
            Document(
                page_content="""Benefits & Limitations of the Trained Classifier:
                - Faster and more consistent than manual binary screening once trained, with no per-image API cost or rate limit.
                - Limitation: it only generalizes to images resembling its training distribution (this casting type, similar lighting and framing) \u2014 unlike a general vision-LLM approach, it will not reliably classify arbitrary, unrelated component photos without retraining on a matching dataset.
                - Because it is binary, "Defective" results still need a secondary step (manual or LLM-assisted) to identify defect subtype for repair planning.""",
                metadata={"category": "benefits_limitations", "source": "benefits_summary"}
            ),
        ]
        self.add_documents(documents)
        print("\u2713 Created sample documents and FAISS index")

    @traceable(name="rag_add_documents", run_type="chain")
    def add_documents(self, documents: List[Document]):
        """Add documents to the vector store."""
        chunks = self.text_splitter.split_documents(documents)
        if self.vector_store is None:
            self.vector_store = FAISS.from_documents(chunks, embeddings)
        else:
            self.vector_store.add_documents(chunks)
        os.makedirs(os.path.dirname(self.persist_directory) or ".", exist_ok=True)
        self.vector_store.save_local(self.persist_directory)

    @traceable(name="rag_retrieve", run_type="retriever")
    def retrieve_context(self, query: str, k: int = TOP_K_RESULTS) -> List[Document]:
        """Retrieve relevant documents for a query."""
        if self.vector_store is None:
            return []
        return self.vector_store.similarity_search(query, k=k)

    @traceable(name="rag_get_context", run_type="chain")
    def get_context_string(self, query: str) -> str:
        """Get context as a formatted string for prompt injection."""
        docs = self.retrieve_context(query)
        if not docs:
            return "No relevant documents found."
        context_parts = []
        for i, doc in enumerate(docs, 1):
            source = doc.metadata.get("source", "Unknown")
            category = doc.metadata.get("category", "General")
            context_parts.append(f"[Document {i} - {category} ({source})]:\n{doc.page_content}\n")
        return "\n".join(context_parts)


# Global RAG instance
rag_system = RAGSystem()

if __name__ == "__main__":
    print("Testing RAG Module...")
    test_queries = [
        "What does the classifier output mean for pass/reject?",
        "What casting defect subtypes should I check for manually?",
        "What should a batch QC report include?",
    ]
    for query in test_queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        context = rag_system.get_context_string(query)
        print(f"Retrieved Context:\n{context}")


In [ ]:
%%writefile router.py
"""Router module for classifying user intent in reverse logistics inspection queries."""
import json
from typing import Dict, Any
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="router_classification", run_type="chain")
def classify_intent(query: str) -> Dict[str, Any]:
    """
    Classify user query into one of the reverse logistics inspection categories.

    Args:
        query: User's question

    Returns:
        Dictionary with classification result and confidence
    """
    system_prompt = """You are an intent classifier for a Reverse Logistics Component Inspection System.
    Classify the user's query into one of these categories:

    1. inspection_explain - Questions about why a specific image or component was classified/flagged a certain way
    2. qc_report - Questions asking for aggregate stats across the current batch (pass/reject counts, category breakdown)
    3. repair_guidance - Questions about how to repair, rework, or handle a flagged component
    4. general - General questions not specific to the above categories

    Respond with JSON format: {"category": "category_name", "confidence": 0.0-1.0, "reasoning": "brief explanation"}
    """

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ]
    )

    try:
        result = json.loads(response.choices[0].message.content)
    except Exception:
        result = {
            "category": "general",
            "confidence": 0.5,
            "reasoning": "Failed to parse response"
        }

    result["usage"] = response.usage.model_dump() if response.usage else None
    return result

@traceable(name="router_decision", run_type="chain")
def route_query(query: str) -> str:
    """
    Route query to appropriate agent based on intent.

    Args:
        query: User's question

    Returns:
        Agent name to route to
    """
    classification = classify_intent(query)
    return classification.get("category", "general")

if __name__ == "__main__":
    print("Testing Router Module...")

    test_queries = [
        "Why was casting_01.jpeg rejected?",
        "How many parts passed in this batch?",
        "How should I repair a part flagged Defective?",
        "What does this system do?"
    ]

    for query in test_queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        classification = classify_intent(query)
        print(f"Classification: {classification}")
        print(f"Routed to: {route_query(query)}")


In [ ]:
%%writefile agents/__init__.py



In [ ]:
%%writefile agents/inspection_agent.py
"""Inspection Explanation Agent for reverse logistics QC."""
from typing import Dict, Any, Optional
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL, TEMPERATURE
from vision_tools import get_batch_inspections

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="inspection_agent", run_type="chain")
def inspection_agent(query: str, batch_id: Optional[str] = None, context: str = None) -> Dict[str, Any]:
    """
    Inspection Agent - Explains why specific images in the current batch were classified as they were.

    Args:
        query: User query about a specific image or set of images
        batch_id: Batch to pull real classifier results from (defaults to the latest batch)
        context: Retrieved RAG context

    Returns:
        Dictionary with response and metadata
    """
    records = get_batch_inspections(batch_id)

    system_prompt = """You are a Casting QC Inspection Expert. Your role is to:
    - Explain why a specific component image was classified Defective or Good Area, quoting the
      trained classifier's confidence score from the tool data, never inventing findings
    - Reference the decision rule (Defective -> Reject/Repair; Good Area -> Pass) when explaining outcomes
    - Remind the user that "Defective" alone does not say which casting defect subtype is present \u2014
      that needs a follow-up manual or secondary check
    - If the user asks about a filename not present in the tool data, say so rather than guessing

    Use the provided tool data (real per-image classifier results from this batch) and context documents to ground your answer."""

    user_prompt = f"""Tool data (this batch's classifier results):
    {records}

    Context from QC guidelines:
    {context if context else "No specific context provided."}

    User Question: {query}

    Please explain the inspection outcome(s)."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return {
        "agent": "inspection_explain",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    print("Testing Inspection Agent...")

    test_query = "What does it mean if a part is flagged Defective?"
    result = inspection_agent(test_query, context="Defective forces a reject; Good Area passes.")

    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")


In [ ]:
%%writefile agents/qc_report_agent.py
"""QC Report Agent for reverse logistics inspection batches."""
from typing import Dict, Any, Optional
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL, TEMPERATURE
from vision_tools import get_batch_summary

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="qc_report_agent", run_type="chain")
def qc_report_agent(query: str, batch_id: Optional[str] = None, context: str = None) -> Dict[str, Any]:
    """
    QC Report Agent - Summarizes pass/reject stats and category breakdown for a batch.

    Args:
        query: User query asking for batch-level stats
        batch_id: Batch to summarize (defaults to the latest batch)
        context: Retrieved RAG context

    Returns:
        Dictionary with response and metadata
    """
    summary = get_batch_summary(batch_id)

    system_prompt = """You are a Casting QC Reporting Analyst. Your role is to:
    - Summarize pass/reject counts and category frequency for the current batch using the real aggregates in the tool data
    - Call out the reject rate and whether it looks unusually high
    - Be precise with numbers from the tool data, never invent figures
    - If the batch is empty, say no inspections have been run yet rather than fabricating numbers

    Use the provided tool data (real aggregated results) and context documents to ground your answer."""

    user_prompt = f"""Tool data (real batch aggregates):
    {summary}

    Context from QC guidelines:
    {context if context else "No specific context provided."}

    User Question: {query}

    Please provide a grounded QC summary."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return {
        "agent": "qc_report",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    print("Testing QC Report Agent...")

    test_query = "Give me a summary of this batch's results."
    result = qc_report_agent(test_query, context="Include pass/reject counts and the reject rate.")

    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")


In [ ]:
%%writefile agents/repair_guidance_agent.py
"""Repair Guidance Agent for reverse logistics QC."""
from typing import Dict, Any, Optional
from openai import OpenAI
from langsmith import traceable
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL, TEMPERATURE
from vision_tools import get_batch_inspections

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

@traceable(name="repair_guidance_agent", run_type="chain")
def repair_guidance_agent(query: str, batch_id: Optional[str] = None, context: str = None) -> Dict[str, Any]:
    """
    Repair Guidance Agent - Recommends next steps for rejected (Defective) components in the current batch.

    Args:
        query: User query about how to handle flagged components
        batch_id: Batch to pull rejected items from (defaults to the latest batch)
        context: Retrieved RAG context

    Returns:
        Dictionary with response and metadata
    """
    records = get_batch_inspections(batch_id)
    rejected = [r for r in records if r.get("decision") == "TIDAK LULUS"]

    system_prompt = """You are a Casting Repair & Rework Expert. Your role is to:
    - Note clearly that the classifier only says "Defective", not which casting defect subtype is present
      (e.g. blow holes/porosity, shrinkage, sand inclusion, cold shut/misrun, surface cracks)
    - List the likely subtypes from the QC guidelines and recommend a manual or secondary check to identify which applies
    - Recommend that any reworked part be re-run through inspection before being marked Good Area
    - If no components were rejected in this batch, say so rather than inventing repair steps

    Use the provided tool data (real rejected items from this batch) and context documents to ground your recommendations."""

    user_prompt = f"""Tool data (rejected items in this batch):
    {rejected}

    Context from QC guidelines:
    {context if context else "No specific context provided."}

    User Question: {query}

    Please provide repair/rework guidance."""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return {
        "agent": "repair_guidance",
        "response": response.choices[0].message.content,
        "usage": response.usage.model_dump() if response.usage else None
    }

if __name__ == "__main__":
    print("Testing Repair Guidance Agent...")

    test_query = "How should we handle the rejected parts in this batch?"
    result = repair_guidance_agent(test_query, context="Defective parts need a secondary check for defect subtype.")

    print(f"\nQuery: {test_query}")
    print(f"\nResponse: {result['response']}")
    print(f"\nToken Usage: {result['usage']}")


In [ ]:
%%writefile graph.py
"""LangGraph workflow for the Reverse Logistics Inspection Assistant."""
from typing import Dict, Any, Literal
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict
from langsmith import traceable

from agents.inspection_agent import inspection_agent
from agents.qc_report_agent import qc_report_agent
from agents.repair_guidance_agent import repair_guidance_agent
from router import classify_intent
from rag import rag_system
from db import db_manager
from openai import OpenAI
from config import OPENAI_API_KEY, GEMINI_BASE_URL, LLM_MODEL

# Initialize OpenAI-compatible client (pointed at Gemini)
client = OpenAI(api_key=OPENAI_API_KEY, base_url=GEMINI_BASE_URL)

# Define state schema with conversation context
class AgentState(TypedDict):
    """State for the LangGraph agent workflow."""
    query: str
    session_id: str
    batch_id: str
    conversation_context: str
    intent: str
    intent_confidence: float
    intent_reasoning: str
    rag_context: str
    agent_response: str
    agent_used: str
    usage: Dict
    error: str

@traceable(name="inject_context", run_type="chain")
def inject_context_node(state: AgentState) -> AgentState:
    """Node to inject RAG context based on intent."""
    try:
        query = state["query"]
        context = state.get("conversation_context", "")

        enhanced_query = query
        if context:
            context_lines = context.split("\n")
            last_user_msg = None
            for line in reversed(context_lines):
                if line.startswith("User:"):
                    last_user_msg = line.replace("User:", "").strip()
                    break
            if last_user_msg:
                enhanced_query = f"Previous question: {last_user_msg}\nCurrent question: {query}"

        state["rag_context"] = rag_system.get_context_string(enhanced_query)
    except Exception as e:
        state["rag_context"] = ""
        state["error"] = f"RAG error: {str(e)}"

    return state

@traceable(name="router_node", run_type="chain")
def router_node(state: AgentState) -> AgentState:
    """Node to classify intent with conversation context."""
    try:
        query = state["query"]
        context = state.get("conversation_context", "")

        enhanced_query = query
        if context:
            context_lines = context.split("\n")
            recent_exchanges = context_lines[-4:] if len(context_lines) > 4 else context_lines
            enhanced_query = f"""Previous conversation:
{chr(10).join(recent_exchanges)}

Current question: {query}"""

        classification = classify_intent(enhanced_query)
        state["intent"] = classification.get("category", "general")
        state["intent_confidence"] = classification.get("confidence", 0.0)
        state["intent_reasoning"] = classification.get("reasoning", "")
        state["usage"] = classification.get("usage")
    except Exception as e:
        state["intent"] = "general"
        state["error"] = f"Router error: {str(e)}"

    return state

def _build_enhanced_context(state: AgentState) -> str:
    rag_context = state["rag_context"]
    conversation_context = state.get("conversation_context", "")
    if conversation_context:
        return f"""Previous conversation context:
{conversation_context}

Relevant documentation:
{rag_context}"""
    return rag_context

@traceable(name="inspection_agent_node", run_type="chain")
def inspection_agent_node(state: AgentState) -> AgentState:
    """Node for the inspection explanation agent."""
    result = inspection_agent(state["query"], state.get("batch_id"), _build_enhanced_context(state))
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="qc_report_agent_node", run_type="chain")
def qc_report_agent_node(state: AgentState) -> AgentState:
    """Node for the QC report agent."""
    result = qc_report_agent(state["query"], state.get("batch_id"), _build_enhanced_context(state))
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="repair_guidance_agent_node", run_type="chain")
def repair_guidance_agent_node(state: AgentState) -> AgentState:
    """Node for the repair guidance agent."""
    result = repair_guidance_agent(state["query"], state.get("batch_id"), _build_enhanced_context(state))
    state["agent_response"] = result["response"]
    state["agent_used"] = result["agent"]
    state["usage"] = result["usage"]
    return state

@traceable(name="general_agent_node", run_type="chain")
def general_agent_node(state: AgentState) -> AgentState:
    """Node for handling general queries with conversation context."""
    query = state["query"]
    context = state.get("conversation_context", "")

    messages = [
        {"role": "system", "content": "You are a helpful assistant for a reverse logistics component inspection system. Provide general information and guide users to specific agents for detailed queries."}
    ]
    if context:
        messages.append({"role": "system", "content": f"Previous conversation:\n{context}"})
    messages.append({"role": "user", "content": query})

    response = client.chat.completions.create(model=LLM_MODEL, messages=messages)

    state["agent_response"] = response.choices[0].message.content
    state["agent_used"] = "general"
    state["usage"] = response.usage.model_dump() if response.usage else None
    return state

@traceable(name="save_to_db", run_type="chain")
def save_to_db_node(state: AgentState) -> AgentState:
    """Node to save conversation to database."""
    try:
        db_manager.save_conversation(
            session_id=state["session_id"],
            user_query=state["query"],
            assistant_response=state["agent_response"],
            agent_used=state["agent_used"],
            metadata={
                "intent": state["intent"],
                "confidence": state["intent_confidence"],
                "batch_id": state.get("batch_id"),
                "conversation_context": state.get("conversation_context", ""),
                "usage": state["usage"]
            }
        )
    except Exception as e:
        state["error"] = f"Database error: {str(e)}"

    return state

def should_continue(state: AgentState) -> Literal["inspection", "qc_report", "repair", "general", END]:
    """Conditional edge to route to appropriate agent."""
    if state.get("error"):
        return END

    intent = state.get("intent", "general")

    if intent == "inspection_explain":
        return "inspection"
    elif intent == "qc_report":
        return "qc_report"
    elif intent == "repair_guidance":
        return "repair"
    else:
        return "general"

def build_inspection_graph():
    """Build and compile the LangGraph workflow."""
    workflow = StateGraph(AgentState)

    workflow.add_node("router", router_node)
    workflow.add_node("inject_context", inject_context_node)
    workflow.add_node("inspection", inspection_agent_node)
    workflow.add_node("qc_report", qc_report_agent_node)
    workflow.add_node("repair", repair_guidance_agent_node)
    workflow.add_node("general", general_agent_node)
    workflow.add_node("save_db", save_to_db_node)

    workflow.set_entry_point("router")
    workflow.add_edge("router", "inject_context")

    workflow.add_conditional_edges(
        "inject_context",
        should_continue,
        {
            "inspection": "inspection",
            "qc_report": "qc_report",
            "repair": "repair",
            "general": "general",
            END: END
        }
    )

    workflow.add_edge("inspection", "save_db")
    workflow.add_edge("qc_report", "save_db")
    workflow.add_edge("repair", "save_db")
    workflow.add_edge("general", "save_db")
    workflow.add_edge("save_db", END)

    return workflow.compile()

# Create global graph instance
inspection_graph = build_inspection_graph()

if __name__ == "__main__":

    ascii_data = inspection_graph.get_graph().draw_ascii()
    print(ascii_data)

    test_queries = [
        ("test_session_1", "What does TIDAK LULUS mean?"),
        ("test_session_1", "How many items passed in this batch?"),
        ("test_session_2", "How should we handle a part flagged Defective?"),
        ("test_session_3", "What does this system do?")
    ]

    for session_id, query in test_queries:
        print(f"\nProcessing: {query}")
        print("-" * 50)

        initial_state = {
            "query": query,
            "session_id": session_id,
            "batch_id": None,
            "conversation_context": "",
            "intent": "",
            "intent_confidence": 0.0,
            "intent_reasoning": "",
            "rag_context": "",
            "agent_response": "",
            "agent_used": "",
            "usage": {},
            "error": ""
        }

        result = inspection_graph.invoke(initial_state)

        print(f"Intent: {result['intent']} (confidence: {result['intent_confidence']:.2f})")
        print(f"Agent Used: {result['agent_used']}")
        print(f"Response: {result['agent_response'][:100]}...")
        print("\u2713 Graph execution complete")


In [ ]:
%%writefile gradio_ui.py
"""
Reverse Logistics Inspection Assistant - UI
Run with: python gradio_ui.py
"""
import uuid
import pandas as pd
import gradio as gr
from vision_tools import classify_batch
from graph import inspection_graph
from db import db_manager

# Store session per user
sessions = {}

def run_inspection(files, batch_id_state):
    """Classify every uploaded image as one batch and show the results."""
    if not files:
        return None, pd.DataFrame(), batch_id_state, "Upload at least one image first."

    image_paths = [f.name if hasattr(f, "name") else f for f in files]
    batch_id = str(uuid.uuid4())
    result = classify_batch(image_paths, batch_id=batch_id)

    gallery_items = []
    rows = []
    for record in result["records"]:
        caption = f"{record['decision']} \u2014 {record['driving_category']} ({record['action']})"
        gallery_items.append((next(p for p in image_paths if p.endswith(record["filename"])), caption))
        rows.append({
            "filename": record["filename"],
            "component_type": record["component_type"],
            "categories_found": ", ".join(record["categories_found"]),
            "decision": record["decision"],
            "action": record["action"],
        })

    df = pd.DataFrame(rows)
    n_pass = sum(1 for r in result["records"] if r["decision"] == "LULUS")
    status = f"Batch {batch_id[:8]}: {len(result['records'])} items inspected, {n_pass} passed, {len(result['records']) - n_pass} rejected."

    return gallery_items, df, batch_id, status

def process_query(query, history, session_id, batch_id):
    """Process a single chat query against the current batch and return response."""

    if not session_id:
        session_id = str(uuid.uuid4())
        sessions[session_id] = []

    conversation_context = ""
    if history:
        context_parts = []
        for msg in history:
            if isinstance(msg, dict) and "role" in msg and "content" in msg:
                role = "User" if msg["role"] == "user" else "Assistant"
                context_parts.append(f"{role}: {msg['content']}")
        conversation_context = "\n".join(context_parts[-6:])

    state = {
        "query": query,
        "session_id": session_id,
        "batch_id": batch_id,
        "conversation_context": conversation_context,
        "intent": "",
        "intent_confidence": 0.0,
        "intent_reasoning": "",
        "rag_context": "",
        "agent_response": "",
        "agent_used": "",
        "usage": {},
        "error": ""
    }

    try:
        result = inspection_graph.invoke(state)

        if result.get("error"):
            response = f"\u274c Error: {result['error']}"
        else:
            agent = result["agent_used"]
            intent = result["intent"]
            confidence = result["intent_confidence"]
            answer = result["agent_response"]

            response = f"""**Agent:** {agent}
**Intent:** {intent} ({confidence:.2f})

{answer}"""

            if result.get("usage"):
                tokens = result["usage"].get("total_tokens", 0)
                response += f"\n\n---\n*Tokens: {tokens}*"

        if history is None:
            history = []

        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": response})

        if session_id in sessions:
            sessions[session_id] = history

        return "", history, session_id

    except Exception as e:
        error_msg = f"\u274c Error: {str(e)}"
        if history is None:
            history = []

        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": error_msg})

        if session_id in sessions:
            sessions[session_id] = history

        return "", history, session_id

def view_history(session_id):
    """View full session history from DB."""
    if not session_id:
        return "No active session"

    history = db_manager.load_session_history(session_id)
    if not history:
        return "No history found"

    output = f"## Session History: {session_id}\n\n"
    for msg in history:
        output += f"**You:** {msg['user_query']}\n\n"
        output += f"**AI ({msg['agent_used']}):** {msg['assistant_response']}\n\n"
        output += "---\n\n"

    return output

def create_session():
    """Create a new session and return session ID."""
    session_id = str(uuid.uuid4())
    sessions[session_id] = []
    return session_id

# Create Gradio interface
with gr.Blocks(title="Reverse Logistics Inspection Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # \U0001F50D Reverse Logistics Inspection Assistant
    Upload one or more casting photos to run defect detection (powered by the classifier trained
    earlier in this notebook), then ask the assistant about specific results, batch-level QC stats,
    or repair guidance.
    """)

    session_state = gr.State(create_session)
    batch_id_state = gr.State(None)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Upload & Inspect")
            file_upload = gr.File(
                label="Component images",
                file_count="multiple",
                file_types=["image"]
            )
            inspect_btn = gr.Button("Run Inspection", variant="primary")
            status_box = gr.Markdown()
            results_gallery = gr.Gallery(label="Detections", columns=3, height=300)
            results_table = gr.Dataframe(
                headers=["filename", "component_type", "categories_found", "decision", "action"],
                label="Batch Results"
            )

        with gr.Column(scale=1):
            gr.Markdown("### 2. Ask the Assistant")
            chatbot = gr.Chatbot(
                label="Conversation",
                height=400,
                type="messages"
            )
            msg = gr.Textbox(label="Your Question", placeholder="e.g., Why was item 2 rejected?")

            with gr.Row():
                submit = gr.Button("Send", variant="primary")
                clear = gr.Button("Clear Chat")

            gr.Markdown("### Sample Queries")
            sample_queries = gr.Dataset(
                components=[msg],
                samples=[
                    ["Give me a summary of this batch's results."],
                    ["Why was the first image flagged?"],
                    ["How should we handle parts flagged Defective?"],
                    ["What's the decision rule for Defective vs Good Area?"]
                ],
                label="Click to try"
            )

            history_btn = gr.Button("View Full History")
            history_output = gr.Markdown()

    def respond(message, chat_history, session_id, batch_id):
        if not message:
            return "", chat_history, session_id
        if chat_history is None:
            chat_history = []
        new_msg, new_history, new_session = process_query(message, chat_history, session_id, batch_id)
        return new_msg, new_history, new_session

    def clear_chat_handler(session_id):
        if session_id in sessions:
            sessions[session_id] = []
        return [], session_id

    inspect_btn.click(
        run_inspection,
        [file_upload, batch_id_state],
        [results_gallery, results_table, batch_id_state, status_box]
    )

    submit.click(respond, [msg, chatbot, session_state, batch_id_state], [msg, chatbot, session_state])
    msg.submit(respond, [msg, chatbot, session_state, batch_id_state], [msg, chatbot, session_state])

    sample_queries.click(lambda x: x[0], [sample_queries], [msg])

    clear.click(clear_chat_handler, [session_state], [chatbot, msg])

    history_btn.click(view_history, [session_state], [history_output])

if __name__ == "__main__":
    demo.launch(
        share=False,
        server_name="127.0.0.1",
        server_port=7860
    )


In [ ]:
# 14. Initialize the database and RAG (FAISS) index
!python db.py
!python vision_tools.py
!python rag.py


## (Optional) Quick smoke test on held-out test images

Runs the real trained classifier (no Gradio needed) on a few images from the test split that the
model never saw during training, as a sanity check before launching the full UI.


In [ ]:
# 15. Smoke-test the pipeline on a handful of held-out test images
import os
from vision_tools import classify_batch

sample_paths = []
for sub in sorted(os.listdir(test_dir))[:2]:
    sub_path = os.path.join(test_dir, sub)
    if os.path.isdir(sub_path):
        files = sorted(os.listdir(sub_path))[:3]
        sample_paths.extend(os.path.join(sub_path, f) for f in files)

if sample_paths:
    result = classify_batch(sample_paths)
    for record in result["records"]:
        print(f"{record['filename']}: {record['decision']} ({record['driving_category']}) -> {record['action']}")
else:
    print("No test images found \u2014 check that test_dir from Part 1 is still set.")


In [ ]:
# 16. Launch the Gradio app
# share=True is required on Colab since 127.0.0.1 isn't reachable from your browser —
# Gradio will print a public *.gradio.live link instead.
from gradio_ui import demo

demo.launch(share=True)


## Notes

- **Trained, not zero-shot:** detection now comes from the MobileNetV2-based classifier trained in
  Part 1 on the Kaggle casting dataset, not a vision-LLM call \u2014 no per-image API cost or rate limit.
- **Label mapping defined after training:** `data/models/label_map.json` records the dataset's raw
  folder-derived class names alongside the domain labels and decision rule chosen in Part 2; change
  that mapping (not the model) if you want a different decision policy.
- **Generalization limit:** the trained classifier only works well on images resembling the casting
  dataset's distribution. To inspect other component types, retrain Part 1 on a matching dataset
  (the rest of the app \u2014 db, rag, agents, graph, UI \u2014 doesn't need to change).
- **Rate limits:** Gemini's free tier (used only by the chat agents and RAG embeddings now) caps
  `gemini-2.5-flash` at 5 requests/minute.
- **Restarting:** if you restart the Colab runtime, re-run all cells from the top \u2014 the working
  directory, trained model, `.env`, database, and FAISS index are all wiped.
- **Stopping the app:** Use Runtime \u2192 Interrupt execution to stop the Gradio server.
- **Updating this notebook:** since this app is embedded directly rather than imported from a
  separate module, any future change to the agent logic needs to be re-applied here too.
